In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import jacfwd
from jax import jit
from jax import vmap
import emcee
import h5py
import corner
import matplotlib.pyplot as plt
import corner
import VBMicrolensing
VBM = VBMicrolensing.VBMicrolensing()
import tqdm
import math
import sys
import multiprocessing
sys.path.append("/moao38_7/nunota/gapmoe/src/gapmoe/")
from gapmoe import gapmoe
import EarthMotion
import parametrics
JD0 = 2450000
from emcee.autocorr import integrated_time

import jax
import jax.numpy as jnp
from jax import jacfwd
from jax import jit
from jax import vmap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib as mpl
from matplotlib.patches import Patch
import arviz as az

import logging
logging.getLogger("emcee.autocorr").setLevel(logging.ERROR)

from matplotlib.lines import Line2D

tref = 10063.874
coords = "17:57:38.03 -28:38:28.53"

VBM.t0_par = tref+JD0
VBM.parallaxsystem = 1
VBM.SetObjectCoordinates(coords)

RA_str,Dec_str = "17:57:38.03","-28:38:28.53"
RA_deg = EarthMotion.hms_string_to_degrees(RA_str)
Dec_deg = EarthMotion.dms_string_to_degrees(Dec_str)
vEarth = EarthMotion.calc_vEarth(tref,RA_deg,Dec_deg)

gapmoe_model = gapmoe(RA_deg,Dec_deg)
gapmoe_model.set_data()

In [2]:
from matplotlib import rcParams
rcParams["font.size"] = 15
rcParams["axes.linewidth"] = 3
rcParams['xtick.top'] = True
rcParams['ytick.right'] = True
rcParams['xtick.direction'] = 'in'
rcParams['ytick.direction'] = 'in'
rcParams['xtick.major.size'] = 10
rcParams['xtick.major.width'] = 1.5
rcParams['xtick.minor.size'] = 5
rcParams['xtick.minor.width'] = 1.5
rcParams['ytick.major.size'] = 10
rcParams['ytick.major.width'] = 1.5
rcParams['ytick.minor.size'] = 5
rcParams['ytick.minor.width'] = 1.5


# mpl.rc('font', **{'family': 'serif', 'serif': ['Liberation Serif', 'DejaVu Serif']})
# plt.style.use('seaborn-v0_8-whitegrid')
# plt.rcParams.update({
#     "font.size": 20,
#     "axes.labelsize": 15,
#     "axes.titlesize": 18,
#     "xtick.labelsize": 12,
#     "ytick.labelsize": 12,
#     "axes.edgecolor": "0.3",
#     "axes.linewidth": 1.4,
#     "grid.alpha": 0.25,
#     "xtick.direction": "in",
#     "ytick.direction": "in",
# })

In [3]:
def calc_ln_prior_kep(theta, thS, vEarth):
    t0, u0, q, ML, DL, DS, murel_N_hel, murel_E_hel,orbital_radi, e, cos_i, Om_NE, om, nu = parametrics.lightcurve_to_physical_kepler(theta, thS, vEarth)

    mu_abs = jnp.sqrt(murel_E_hel**2 + murel_N_hel**2)
    mu_phi = jnp.arctan2(murel_E_hel, murel_N_hel)
    DL *= 1e3
    DS *= 1e3

    if not (10000 <= t0 <= 11000):
        return -np.inf
    if not (-5 <= u0 <= 5):
        return -np.inf
    if not (1e-5 <= q <= 1.0):
        return -np.inf
    if not (0 <= e <= 1.0):
        return -np.inf
    if not (-np.pi <= Om_NE <= np.pi):
        return -np.inf
    if not (-np.pi <= om <= np.pi):
        return -np.inf
    if not (-np.pi <= nu <= np.pi):
        return -np.inf
    if not (0 <= orbital_radi <= 1e4):
        return -np.inf

    if not (gapmoe_model.M_MIN <= ML <= gapmoe_model.M_MAX):
        return -np.inf
    if not (gapmoe_model.DL_MIN <= DL < gapmoe_model.DL_MAX):
        return -np.inf
    if not (gapmoe_model.DS_MIN <= DS < gapmoe_model.DS_MAX):
        return -np.inf
    if DL >= DS:
        return -np.inf
    if not (gapmoe_model.MU_MIN <= mu_abs < gapmoe_model.MU_MAX):
        return -np.inf

    return gapmoe_model.log_galactic_prior(ML, DL, DS, mu_abs, mu_phi) - np.log(mu_abs) #jacobian for (muN, muE) -> (mu,phi)

In [4]:
def read_chain(path,burnin,thin,flat=True):
    sampler = emcee.backends.HDFBackend(path)
    chain = sampler.get_chain(flat=flat, discard=burnin, thin=thin)
    blob = sampler.get_blobs(flat=flat, discard=burnin, thin=thin) 
    return chain, blob

In [5]:
kep_gap_chain_01,kep_gap_blob_01 = read_chain("../backup_important/rerun_simu_01_kepler_chain_01.h5",0,1,flat=False)
kep_wo_chain_01,kep_wo_blob_01 = read_chain("../backup_important/rerun_wo_gap_simu_01_kepler_chain_01.h5",0,1,flat=False)

kep_gap_chain_02, kep_gap_blob_02 = read_chain("../backup_important/rerun_simu_02_kepler_chain_01.h5",0,1,flat=False)
kep_wo_chain_02, kep_wo_blob_02 = read_chain("../backup_important/rerun_wo_gap_simu_02_kepler_chain_01.h5",0,1,flat=False)

In [6]:
kep_gap_chain_01_add, kep_gap_blob_01_add = read_chain("../test_result/rogue1/backend/rerun_simu_01_kepler_chain_01.h5",0,1,flat=False)
kep_wo_chain_01_add,kep_wo_blob_01_add = read_chain("../test_result/rogue1/backend/rerun_wo_gap_simu_01_kepler_chain_01.h5",0,1,flat=False)

kep_gap_chain_02_add, kep_gap_blob_02_add = read_chain("../test_result/rogue1/backend/rerun_simu_02_kepler_chain_01.h5",0,1,flat=False)
kep_wo_chain_02_add,kep_wo_blob_02_add = read_chain("../test_result/rogue1/backend/rerun_wo_gap_simu_02_kepler_chain_01.h5",0,1,flat=False)

In [7]:
kep_gap_chain_01_all = np.concatenate([kep_gap_chain_01, kep_gap_chain_01_add], axis=0)[:250000,:28]
kep_wo_chain_01_all  = np.concatenate([kep_wo_chain_01,  kep_wo_chain_01_add],  axis=0)[:250000,:28]

kep_gap_chain_02_all = np.concatenate([kep_gap_chain_02, kep_gap_chain_02_add], axis=0)[:250000,:28]
kep_wo_chain_02_all  = np.concatenate([kep_wo_chain_02,  kep_wo_chain_02_add],  axis=0)[:250000,:28]

kep_gap_blob_01_all = np.concatenate([kep_gap_blob_01, kep_gap_blob_01_add], axis=0)[:250000,:28]
kep_wo_blob_01_all  = np.concatenate([kep_wo_blob_01,  kep_wo_blob_01_add],  axis=0)[:250000,:28]

kep_gap_blob_02_all = np.concatenate([kep_gap_blob_02, kep_gap_blob_02_add], axis=0)[:250000,:28]
kep_wo_blob_02_all  = np.concatenate([kep_wo_blob_02,  kep_wo_blob_02_add],  axis=0)[:250000,:28]

In [8]:
print(kep_gap_chain_02_add.shape, kep_gap_chain_02.shape)

(131544, 29, 14) (129000, 29, 14)


In [9]:
def calc_tau(arr):
    taus = []
    for pi in range(arr.shape[2]):
        tau_walkers = []
        for w in range(arr.shape[1]):
            chain = arr[:, w, pi]
            tau = emcee.autocorr.integrated_time(chain[:, None], quiet=True)
            tau_walkers.append(tau.item())
        mean_tau = np.mean(tau_walkers)
        taus.append(mean_tau)
        print(f"param[{pi}] mean τ_int over {arr.shape[1]} walkers = {mean_tau:.2f}")

    return max(taus)
    
def tau_progress(arr, step=10000):
    nsteps, nwalkers, nparams = arr.shape
    steps = []
    tau_matrix = []

    for n in range(step, nsteps+1, step):
        taus = []
        for pi in range(nparams):
            tau_walkers = []
            for w in range(nwalkers):
                chain = arr[:n, w, pi]
                tau = emcee.autocorr.integrated_time(chain[:, None], quiet=True)
                tau_walkers.append(tau.item())
            mean_tau = np.mean(tau_walkers)
            taus.append(mean_tau)
        steps.append(n)
        tau_matrix.append(taus)

    return np.array(steps), np.array(tau_matrix)


In [10]:
tau_gap_01 = calc_tau(kep_gap_chain_01_all)
tau_gap_02 = calc_tau(kep_gap_chain_02_all)
tau_wo_01 = calc_tau(kep_wo_chain_01_all)
tau_wo_02 = calc_tau(kep_wo_chain_02_all)

param[0] mean τ_int over 28 walkers = 978.98
param[1] mean τ_int over 28 walkers = 1033.79
param[2] mean τ_int over 28 walkers = 1023.54
param[3] mean τ_int over 28 walkers = 875.16
param[4] mean τ_int over 28 walkers = 993.65
param[5] mean τ_int over 28 walkers = 1039.00
param[6] mean τ_int over 28 walkers = 1153.59
param[7] mean τ_int over 28 walkers = 1045.78
param[8] mean τ_int over 28 walkers = 1082.52
param[9] mean τ_int over 28 walkers = 1300.80
param[10] mean τ_int over 28 walkers = 1902.96
param[11] mean τ_int over 28 walkers = 1804.17
param[12] mean τ_int over 28 walkers = 2311.12
param[13] mean τ_int over 28 walkers = 2892.59
param[0] mean τ_int over 28 walkers = 855.40
param[1] mean τ_int over 28 walkers = 1675.27
param[2] mean τ_int over 28 walkers = 1757.14
param[3] mean τ_int over 28 walkers = 891.69
param[4] mean τ_int over 28 walkers = 1513.65
param[5] mean τ_int over 28 walkers = 787.78
param[6] mean τ_int over 28 walkers = 1108.11
param[7] mean τ_int over 28 walkers 

In [11]:
# steps, tau_mat_gap_01 = tau_progress(kep_gap_chain_01_all, step=10000)
# steps, tau_mat_gap_02 = tau_progress(kep_gap_chain_02_all, step=10000)
# steps, tau_mat_wo_01 = tau_progress(kep_wo_chain_01_all, step=10000)
# steps, tau_mat_wo_02 = tau_progress(kep_wo_chain_02_all, step=10000)

In [12]:
# params = ["t0","tE","u0","rho","q","s","alpha","piEN","piEE","g1","g2","g3","rs","as"]
# for i in range(tau_mat_gap_01.shape[1]):
#     plt.figure(figsize=(7,3))
#     plt.plot(steps, tau_mat_gap_01[:,i], label="gap_01")
#     plt.plot(steps, tau_mat_gap_02[:,i], label="gap_02")
#     plt.plot(steps, tau_mat_wo_01[:,i], label="wo_01")
#     plt.plot(steps, tau_mat_wo_02[:,i], label="wo_02")
    
#     plt.plot(steps, steps/50, "k--")
    
#     plt.xlabel("steps")
#     plt.ylabel("tau")
#     plt.title(f"{params[i]}")
#     plt.legend(fontsize=10)
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()

In [13]:
# params = ["t0","tE","u0","rho","q","s","alpha","piEN","piEE","g1","g2","g3","rs","as"]
# colors = plt.cm.tab20(np.linspace(0, 1, len(params)))  # 14 色生成


# plt.figure(figsize=(7,6))

# for i in range(tau_mat_gap_01.shape[1]):
#     plt.plot(steps, tau_mat_gap_01[:,i], color=colors[i], label=params[i])

# plt.plot(steps, steps/50, "k--")

# plt.xscale("log")
# plt.yscale("log")
# plt.ylim(2e2,1e4)
# plt.xlabel("steps")
# plt.ylabel("tau")
# plt.legend(fontsize=10)
# plt.tight_layout()
# plt.show()

In [14]:
kep_gap_chain_01_thinned =  kep_gap_chain_01_all[1000:].reshape(-1, 14)[::int(tau_gap_01)+1]
kep_gap_chain_02_thinned =  kep_gap_chain_02_all[1000:].reshape(-1, 14)[::int(tau_gap_02)+1]
kep_wo_chain_01_thinned =  kep_wo_chain_01_all[1000:].reshape(-1, 14)[::int(tau_wo_01)+1]
kep_wo_chain_02_thinned =  kep_wo_chain_02_all[1000:].reshape(-1, 14)[::int(tau_wo_02)+1]

kep_gap_blob_01_thinned =  kep_gap_blob_01_all[1000:].reshape(-1)[::int(tau_gap_01)+1]
kep_gap_blob_02_thinned =  kep_gap_blob_02_all[1000:].reshape(-1)[::int(tau_gap_02)+1]
kep_wo_blob_01_thinned =  kep_wo_blob_01_all[1000:].reshape(-1)[::int(tau_wo_01)+1]
kep_wo_blob_02_thinned =  kep_wo_blob_02_all[1000:].reshape(-1)[::int(tau_wo_02)+1]

In [15]:
vmap_kep = jax.vmap(
    lambda theta, thS: parametrics.lightcurve_to_physical_kepler(theta, thS, vEarth),
    in_axes=(0, 0)
)

vmap_det_kep = jax.vmap(
    lambda theta, thS: parametrics.calc_ln_det_jacobian_kepler(theta, thS, vEarth),
    in_axes=(0, 0)
)

kep_gap_phys_01 = np.array(vmap_kep(kep_gap_chain_01_thinned,kep_gap_blob_01_thinned))
kep_wo_phys_01 = np.array(vmap_kep(kep_wo_chain_01_thinned, kep_wo_blob_01_thinned))
kep_wo_det_01 = np.array(vmap_det_kep(kep_wo_chain_01_thinned, kep_wo_blob_01_thinned))
kep_wo_gal_01 = np.array(list(map(
    lambda args: calc_ln_prior_kep(args[0], args[1], vEarth),
    zip(kep_wo_chain_01_thinned, kep_wo_blob_01_thinned)
)))

kep_gap_phys_02 = np.array(vmap_kep(kep_gap_chain_02_thinned,kep_gap_blob_02_thinned))
kep_wo_phys_02 = np.array(vmap_kep(kep_wo_chain_02_thinned, kep_wo_blob_02_thinned))
kep_wo_det_02 = np.array(vmap_det_kep(kep_wo_chain_02_thinned, kep_wo_blob_02_thinned))
kep_wo_gal_02 = np.array(list(map(
    lambda args: calc_ln_prior_kep(args[0], args[1], vEarth),
    zip(kep_wo_chain_02_thinned, kep_wo_blob_02_thinned)
)))


weight_kep_01 = np.exp((kep_wo_det_01 + kep_wo_gal_01) - np.max(kep_wo_det_01 + kep_wo_gal_01))
weight_kep_01 /= np.sum(weight_kep_01)

weight_kep_02 = np.exp((kep_wo_det_02 + kep_wo_gal_02) - np.max(kep_wo_det_02 + kep_wo_gal_02))
weight_kep_02 /= np.sum(weight_kep_02)

In [16]:
Neff_gap_01 = kep_gap_chain_01_thinned.shape[0]
Neff_gap_02= kep_gap_chain_02_thinned.shape[0]
Neff_wo_01_unw = kep_wo_chain_01_thinned.shape[0]
Neff_wo_02_unw= kep_wo_chain_02_thinned.shape[0]

Neff_wo_01 = (np.sum(weight_kep_01)**2) / np.sum(weight_kep_01**2)
Neff_wo_02 = (np.sum(weight_kep_02)**2) / np.sum(weight_kep_02**2)

In [17]:
# np.save("../test_result/array/kep_gap_chain_01",kep_gap_chain_01_thinned)
# np.save("../test_result/array/kep_gap_chain_02",kep_gap_chain_02_thinned)
# np.save("../test_result/array/kep_wo_chain_01",kep_wo_chain_01_thinned)
# np.save("../test_result/array/kep_wo_chain_02",kep_wo_chain_02_thinned)

# np.save("../test_result/array/kep_gap_phys_01",kep_gap_phys_01)
# np.save("../test_result/array/kep_gap_phys_02",kep_gap_phys_02)
# np.save("../test_result/array/kep_wo_phys_01",kep_wo_phys_01)
# np.save("../test_result/array/kep_wo_phys_02",kep_wo_phys_02)

# np.save("../test_result/array/kep_wo_wt_01",weight_kep_01)
# np.save("../test_result/array/kep_wo_wt_02",weight_kep_02)
# np.save("../test_result/array/kep_wo_brob_01",kep_wo_blob_01_thinned)
# np.save("../test_result/array/kep_wo_brob_02",kep_wo_blob_02_thinned)

In [18]:
print(Neff_gap_01)
print(Neff_gap_02)
print(Neff_wo_01_unw)
print(Neff_wo_02_unw)

print(Neff_wo_01)
print(Neff_wo_02)


2410
2957
1507
1409
2.6080516478712688
5.533859481692813


In [19]:
tau_wo_02*50

np.float64(247408.0285952548)

In [20]:
# para_wo_gap_chain_01,para_wo_gap_blob_01 = read_chain("../test_result/chain/wo_",0,1,flat=False)
# tau_wo_gap_para_01 = calc_tau(para_wo_gap_chain_01)
# para_wo_gap_chain_01_thinned =  para_wo_gap_chain_01[1000:].reshape(-1, 9)[::int(tau_wo_gap_para_01)+1]
# para_wo_gap_blob_01_thinned =  para_wo_gap_blob_01[1000:].reshape(-1)[::int(tau_wo_gap_para_01)+1]


In [21]:
# np.save("../test_result/array/para_event01",para_wo_gap_chain_01_thinned)
# np.save("../test_result/array/para_event01_blob",para_wo_gap_blob_01_thinned)

In [22]:
# static_wo_gap_chain_01,static_wo_gap_blob_01 = read_chain("../test_result/rogue1/backend/rerun_wo_gap_simu_01_static_chain_01.h5",0,1,flat=False)
# tau_wo_gap_static_01 = calc_tau(static_wo_gap_chain_01)
# static_wo_gap_chain_01_thinned =  static_wo_gap_chain_01[1000:].reshape(-1, 7)[::int(tau_wo_gap_static_01)+1]
# static_wo_gap_blob_01_thinned =  static_wo_gap_blob_01[1000:].reshape(-1)[::int(tau_wo_gap_static_01)+1]

In [23]:
# np.save("../test_result/array/static_wo_chain_01",static_wo_gap_chain_01_thinned )
# np.save("../test_result/array/static_wo_brob_01",static_wo_gap_blob_01_thinned)

In [25]:
static_wo_gap_chain_02,static_wo_gap_blob_02 = read_chain("../test_result/rogue1/backend/rerun_wo_gap_simu_01_static_chain_01.h5",0,1,flat=False)
tau_wo_gap_static_02 = calc_tau(static_wo_gap_chain_02)
static_wo_gap_chain_02_thinned =  static_wo_gap_chain_02[1000:].reshape(-1, 7)[::int(tau_wo_gap_static_02)+1]
static_wo_gap_blob_02_thinned =  static_wo_gap_blob_02[1000:].reshape(-1)[::int(tau_wo_gap_static_02)+1]


param[0] mean τ_int over 30 walkers = 2588.48
param[1] mean τ_int over 30 walkers = 2024.05
param[2] mean τ_int over 30 walkers = 2430.77
param[3] mean τ_int over 30 walkers = 1375.30
param[4] mean τ_int over 30 walkers = 2557.57
param[5] mean τ_int over 30 walkers = 2549.41
param[6] mean τ_int over 30 walkers = 2465.07


In [29]:
print(tau_gap_01)
print(tau_wo_gap_static_02)

print(28*250000/tau_gap_01)
print(28*250000/tau_wo_gap_static_02)

2892.588556669757
2588.4767333070517
2419.9777683069847
2704.293189089926


In [ ]:
# np.save("../test_result/array/static_wo_chain_02",static_wo_gap_chain_02_thinned )
# np.save("../test_result/array/static_wo_brob_02",static_wo_gap_blob_02_thinned)

In [ ]:
label_inds_kep = {"t0": 0, "tE": 1, "u0": 2, "rho": 3, "q": 4, "s": 5, "alpha": 6,
                  "piEN": 7, "piEE": 8, "gamma1": 9, "gamma2": 10, "gamma3": 11,
                  "rs": 12, "as": 13}

label_inds_kep_phys = {"ML": 3, "DL": 4, "DS": 5, "muN": 6, "muE": 7, "a": 8,
                       "e": 9, "cos_i": 10, "Omega": 11, "omega": 12, "nu": 13}

In [ ]:
for i in range(kep_gap_chain_01_all.shape[1]):
    plt.figure(figsize=(6,3))
    plt.plot(kep_gap_chain_01_all[:,i,-3],alpha=0.5)
    plt.plot(kep_wo_chain_01_all[:,i,-3],alpha=0.5)
#     plt.ylim(0,15)
    plt.title(f"{i}")
    plt.show()


In [ ]:
for i in range(kep_gap_chain_01_all.shape[1]):
    plt.figure(figsize=(6,3))
    plt.plot(kep_gap_chain_02_all[:,i,-3],alpha=0.5)
    plt.plot(kep_wo_chain_02_all[:,i,-3],alpha=0.5)
#     plt.ylim(0,15)
    plt.title(f"{i}")
    plt.show()


In [ ]:
corner.corner(kep_gap_chain_01_thinned,labels=list(label_inds_kep.keys()))

plt.show()